In [8]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

from config.spark import spark


In [9]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

customer_schema = StructType([
    StructField("customer_id",StringType()),
    StructField("customer_unique_id",StringType()),
    StructField("customer_zip_code_prefix",IntegerType()),
    StructField("customer_city",StringType()),
    StructField("customer_state",StringType())
])


In [10]:
customers_df = (
    spark.read.format('csv')
    .option('header','True')
   # .option('inferschema','True')
    .schema(customer_schema)
   
    .load('../../landing/files/olist_customers_dataset.csv')
    
)

In [11]:
from pyspark.sql.functions import current_timestamp

customers_df = customers_df.withColumn("ingestion_time",current_timestamp())

In [12]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [13]:
customers_df.write.mode("overwrite").saveAsTable("bronze.customers")

In [14]:
spark.sql(
    """select * from bronze.customers
    """
).show()

+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|      ingestion_time|
+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+
|503840d4f2a1a7609...|ffc4233210eac4ec1...|                   14811|          araraquara|            SP|2026-10-19 15:26:...|
|52e73a5d0a1d4c56b...|b43530186123fb6d9...|                   62625|               missi|            CE|2026-10-19 15:26:...|
|16cb62869f9719571...|c3cc321141423ab8a...|                   55560|           barreiros|            PE|2026-10-19 15:26:...|
|4979ba0e6037e4b28...|80768413a59684f1e...|                   29307|cachoeiro de itap...|            ES|2026-10-19 15:26:...|
|11ec4bc0610184925...|bd836cf4fce7f808b...|                   22420|      rio de janeiro|            RJ|2026-10-19 15: